 A small anatomical pipeline (solution)

Start from the Styx book's [anatomical preprocessing example](https://niwrap.dev/styxbook/examples/anat_preproc.html) (reorient, crop, BET, FAST), then add N4 bias correction, registration to MNI152NLin2009cAsym, the segmentation in template space, and a loop over subjects.

In [ ]:
import logging
import os
import shutil
from pathlib import Path

import niwrap
from nilearn import plotting
from niwrap import ants, fsl

BIDS = Path("../data/ds000001")
TEMPLATE = Path("../data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz")
SPACE = "MNI152NLin2009cAsym"
OUT = Path("../derivatives/workshop")

RUNNER = "auto"  # change to "dry" if Docker doesn't work on your machine
if RUNNER == "dry":
    runner = niwrap.use_dry()
else:
    runner = niwrap.use_auto(
        data_dir="../work/03",
        environ={"ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS": str(os.cpu_count())},
    )
    logging.getLogger(runner.logger_name).setLevel(logging.INFO)

`environ` is passed into every container. Without `ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS`, ANTs registration uses one core.

# Step 0: the book's example

The book's `anatomical_preprocessing`, updated for NiWrap 1.1: `fslreorient2std`, `robustfov` and `fast` need named outputs, or their output attributes are `None`. BET's `robust_iters` (`-R`) is dropped because it fails in the default FSL image; `robustfov` already removes the neck.

In [ ]:
def anatomical_preprocessing(t1w: Path) -> tuple[fsl.BetOutputs, fsl.FastOutputs]:
    reorient = fsl.fslreorient2std(input_image=t1w, output_image="t1w_reoriented.nii.gz")
    crop = fsl.robustfov(input_file=reorient.output_image, output_image="t1w_cropped.nii.gz")
    brain = fsl.bet(infile=crop.output_roi_volume, fractional_intensity=0.5, binary_mask=True)
    seg = fsl.fast(in_files=[brain.outfile], img_type=1, out_basename="t1w")
    return brain, seg


brain, seg = anatomical_preprocessing(BIDS / "sub-01/anat/sub-01_T1w.nii.gz")
seg.tissue_class_map

In [ ]:
if RUNNER != "dry":
    plotting.plot_roi(seg.tissue_class_map, bg_img=brain.outfile, title="sub-01 FAST")

# Step 1: bias-field correction

Run ANTs N4 between BET and FAST. Its output name is built with a helper, `ants.n4_bias_field_correction_corrected_output_params(...)`.

In [ ]:
def anatomical_preprocessing(t1w: Path) -> tuple[fsl.BetOutputs, Path, fsl.FastOutputs]:
    reorient = fsl.fslreorient2std(input_image=t1w, output_image="t1w_reoriented.nii.gz")
    crop = fsl.robustfov(input_file=reorient.output_image, output_image="t1w_cropped.nii.gz")
    brain = fsl.bet(infile=crop.output_roi_volume, fractional_intensity=0.5, binary_mask=True)
    n4 = ants.n4_bias_field_correction(
        input_image=brain.outfile,
        mask_image=brain.binary_mask,
        image_dimensionality=3,
        output=ants.n4_bias_field_correction_corrected_output_params(
            corrected_output_file_name="t1w_n4.nii.gz"
        ),
    )
    seg = fsl.fast(in_files=[n4.output_image_outfile], img_type=1, out_basename="t1w")
    return brain, n4.output_image_outfile, seg

# Step 2: register to the template

`ants.ants_registration_sy_n_quick_sh` runs `antsRegistrationSyNQuick.sh` (rigid, affine, SyN). Outputs include the warped image, the affine (`output_transform`) and the warp (`output_warp`).

In [ ]:
def register(t1w_brain: Path) -> ants.AntsRegistrationSyNQuickShOutputs:
    return ants.ants_registration_sy_n_quick_sh(
        dimensionality=3,
        fixed_image=TEMPLATE,
        moving_image=t1w_brain,
        output_prefix="t1w_to_template_",
    )

# Step 3: warp the segmentation

Use `ants.ants_apply_transforms` with `GenericLabel` interpolation so labels stay integers. ANTs applies transforms last to first: list the warp before the affine.

In [ ]:
def warp_labels(labels: Path, xfm: ants.AntsRegistrationSyNQuickShOutputs) -> Path:
    warped = ants.ants_apply_transforms(
        dimensionality=3,
        input_image=labels,
        reference_image=TEMPLATE,
        interpolation="GenericLabel",
        transform=[
            ants.ants_apply_transforms_transform_file_name_params(xfm.output_warp),
            ants.ants_apply_transforms_transform_file_name_params(xfm.output_transform),
        ],
        output=ants.ants_apply_transforms_warped_output_params("dseg_template.nii.gz"),
    )
    return warped.output_image_outfile

# Step 4: one subject, start to finish

`process_subject` chains the steps and returns the files to keep. `save` copies them into `derivatives/` with BIDS names.

In [ ]:
def process_subject(t1w: Path) -> dict[str, Path]:
    brain, t1w_brain, seg = anatomical_preprocessing(t1w)
    xfm = register(t1w_brain)
    return {
        "desc-preproc_T1w.nii.gz": t1w_brain,
        "desc-brain_mask.nii.gz": brain.binary_mask,
        "dseg.nii.gz": seg.tissue_class_map,
        f"space-{SPACE}_desc-preproc_T1w.nii.gz": xfm.output_warped_image,
        f"space-{SPACE}_dseg.nii.gz": warp_labels(seg.tissue_class_map, xfm),
        f"from-T1w_to-{SPACE}_mode-image_xfm.mat": xfm.output_transform,
        f"from-T1w_to-{SPACE}_mode-image_xfm.nii.gz": xfm.output_warp,
    }


def save(outputs: dict[str, Path], subject: str) -> dict[str, Path]:
    out_dir = OUT / subject / "anat"
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = {}
    for suffix, src in outputs.items():
        saved[suffix] = out_dir / f"{subject}_{suffix}"
        if RUNNER != "dry":
            shutil.copy2(src, saved[suffix])
    return saved

In [ ]:
sub01 = save(process_subject(BIDS / "sub-01/anat/sub-01_T1w.nii.gz"), "sub-01")
sub01

In [ ]:
if RUNNER != "dry":
    plotting.plot_roi(sub01[f"space-{SPACE}_dseg.nii.gz"], bg_img=TEMPLATE, title="sub-01 tissue classes, MNI space")

# Step 5: every subject

A loop over subjects. sub-01 is already done.

In [ ]:
results = {"sub-01": sub01}
for t1w in sorted(BIDS.glob("sub-*/anat/sub-*_T1w.nii.gz")):
    subject = t1w.name.split("_")[0]
    if subject not in results:
        print(f"Processing {subject}")
        results[subject] = save(process_subject(t1w), subject)

In [ ]:
if RUNNER != "dry":
    for subject, files in results.items():
        plotting.plot_anat(files[f"space-{SPACE}_desc-preproc_T1w.nii.gz"], title=subject, display_mode="z", cut_coords=5)

# Next: more subjects

For more than a few subjects, a workflow engine runs them in parallel, re-runs only what's missing, and submits to a cluster. Next section.